In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import anndata as ad

# Configure visualization environment for scalable high-resolution vector exports
scv.settings.set_figure_params('scvelo')

# ==============================================================================
# 1. PARAMETERS & PATH SPECIFICATION
# ==============================================================================
path_obs = "/mnt/d/....../metadata_obs.csv"
path_umap = "/mnt/d/.......metadata_umap.csv"
path_velo = "/mnt/d/......./adata_raw_combined_fixed.h5ad"
save_path = "/mnt/d/.........scVelo/adata_final_aligned_velocity.h5ad"

print("--- [Initialization] Target Environment Check ---")
print(f"NumPy Version: {np.__version__}")
print(f"Scanpy Version: {sc.__version__}")
print(f"Anndata Version: {ad.__version__}")

# ==============================================================================
# 2. FILE INGESTION & DIAGNOSTIC QUALITY CONTROL
# ==============================================================================
print("\n--- [Data Ingestion] Reading Metadata Tables and Velocity Matrices ---")
obs_df = pd.read_csv(path_obs, index_col=0)
umap_df = pd.read_csv(path_umap, index_col=0)
adata_velo = sc.read_h5ad(path_velo)

print(f"Available structural features in metadata: {obs_df.columns.tolist()}")

# Verify target viral load and covariate mapping indicators
viral_candidates = ['ZIKV', 'zikv', 'bingdu', 'pct_counts_zikv', 'viral_load']
for col in viral_candidates:
    if col in obs_df.columns:
        valid_hits = obs_df[obs_df[col] > 0][col].head(5).tolist() if obs_df[col].dtype != 'object' else obs_df[col].unique()[:5]
        print(f"  > Target track detected: '{col}' | Type: {obs_df[col].dtype} | Sample Head: {valid_hits}")
    else:
        print(f"  > Target track absent: '{col}'")

# ==============================================================================
# 3. HIGH-PRECISION BARCODE ALIGNMENT
# ==============================================================================
print("\n--- [Index Processing] Synchronizing Single-Cell Barcodes ---")
print(f"Discovered sample pools: {obs_df['Sample'].unique().tolist()}")

# Harmonize index architecture: Map unique cohort prefix with isolated cell barcode hashes
obs_df.index = [f"{row['Sample']}_{idx.split('-')[0]}" for idx, row in obs_df.iterrows()]
umap_df.index = obs_df.index

# Evaluate dataset overlap matrix intersection
common_cells = adata_velo.obs_names.intersection(obs_df.index)
print(f"Successful matrix intersection yield: {len(common_cells)} unified cells identified.")

if len(common_cells) > 0:
    # Slice velocity matrix array by filtered index matches
    adata = adata_velo[common_cells].copy()
    
    # Isolate and clean target categorical/continuous observation attributes
    keep_cols = ['Sample', 'cell_type', 'pct_counts_zikv', 'Group', 'leiden']
    adata.obs = obs_df.loc[common_cells, keep_cols].copy()
    
    # Enforce standard scientific terminology mapping for viral data layers
    adata.obs.rename(columns={'pct_counts_zikv': 'ZIKV'}, inplace=True)
    
    # Assign dimension spatial embedding arrays
    adata.obsm['X_umap'] = umap_df.loc[common_cells].values
    print("Metadata tracking tracks and dimensional spatial matrices successfully bound.")

    # ==============================================================================
    # 4. EMBEDDING TOPOLOGY VALIDATION
    # ==============================================================================
    # Export verification plots mapping topology, viral accumulation vectors, and sample distribution
    sc.pl.umap(
        adata, 
        color=['cell_type', 'ZIKV', 'Sample'], 
        ncols=3, 
        frameon=False, 
        palette='turbo',
        show=False
    )
    
    # Commit locked object tracking frame to persistent local storage
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    adata.write(save_path)
    print(f"\n[SUCCESS] Pipeline finalized. Matrix safely written to: {save_path}")
    
    # Inspect final clinical sample footprint block structure
    print("\n--- [Matrix Check] Sliced Observation Attribute Space (adata.obs) ---")
    print(adata.obs.head())
else:
    print("\n[CRITICAL ERROR] Index intersection is empty. Alignment aborted.")
    if len(obs_df.index) > 0 and len(adata_velo.obs_names) > 0:
        print(f" -> Sample footprint from reference metadata index: {obs_df.index[0]}")
        print(f" -> Sample footprint from source velocity index:  {adata_velo.obs_names[0]}")